In [40]:
import pandas as pd
import numpy as np
from datetime import date



In [41]:
# Step 1 : Load Data
df = pd.read_csv('../data/personal_transactions.csv', parse_dates=['Date'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')

In [42]:
print (df.shape) 
df.head(10)

(806, 6)


,date,description,amount,transaction_type,category,account_name
0,2018-01-01,Amazon,11.11,debit,Shopping,Platinum Card
1,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking
2,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,Silver Card
3,2018-01-03,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card
4,2018-01-04,Netflix,11.76,debit,Movies & DVDs,Platinum Card
5,2018-01-05,American Tavern,25.85,debit,Restaurants,Silver Card
6,2018-01-06,Hardware Store,18.45,debit,Home Improvement,Silver Card
7,2018-01-08,Gas Company,45.00,debit,Utilities,Checking
8,2018-01-08,Hardware Store,15.38,debit,Home Improvement,Silver Card
9,2018-01-09,Spotify,10.69,debit,Music,Platinum Card


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806 entries, 0 to 805
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              806 non-null    datetime64[ns]
 1   description       806 non-null    object        
 2   amount            806 non-null    float64       
 3   transaction_type  806 non-null    object        
 4   category          806 non-null    object        
 5   account_name      806 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 37.9+ KB


In [44]:
# create month column
df['date'] = pd.to_datetime(df['date'])

print(df['date'].dtype)

datetime64[ns]


In [45]:
# Extract month from date
df['month'] = df['date'].dt.to_period('M')

In [46]:
df.head()

,date,description,amount,transaction_type,category,account_name,month
0,2018-01-01,Amazon,11.11,debit,Shopping,Platinum Card,2018-01
1,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking,2018-01
2,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,Silver Card,2018-01
3,2018-01-03,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2018-01
4,2018-01-04,Netflix,11.76,debit,Movies & DVDs,Platinum Card,2018-01


In [48]:
df['account_name'].unique()

array(['Platinum Card', 'Checking', 'Silver Card'], dtype=object)

In [50]:
# Fix sign logic based on transaction type and account
def fix_sign(row):
    t = str(row['transaction_type']).strip().lower()
    amt = abs(row['amount'])
    
    # Credit = money IN = positive
    # Debit = money OUT = negative
    return amt if t == 'credit' else -amt
df['amount_clean'] = df.apply(fix_sign, axis=1)

In [53]:
print("Income total: $", df[df['amount_clean'] > 0]['amount_clean'].sum().round(2))
print("Expense total: $", df[df['amount_clean'] < 0]['amount_clean'].sum().round(2))
print("Income total: $", df['amount_clean'].sum().round(2))

Income total: $ 124269.76
Expense total: $ -96083.78
Income total: $ 28185.98


In [56]:
# Monthly summary
monthly = df.resample('M', on='date').agg(
    income = ('amount_clean', lambda x: x[x > 0].sum()),
    expense = ('amount_clean', lambda x: x[x < 0].sum()),
    net = ('amount_clean', 'sum')
).round(2)

In [59]:
print(monthly)
monthly.to_csv('data/monthly_cashflow.csv')

             income   expense      net
date                                  
2018-01-31  7162.89  -2931.45  4231.44
2018-02-28  5220.75  -3165.05  2055.70
2018-03-31  7321.50  -3500.16  3821.34
2018-04-30  7166.88  -6029.54  1137.34
2018-05-31  5091.55 -11392.03 -6300.48
2018-06-30  6017.19  -3665.88  2351.31
2018-07-31  4666.34  -2968.98  1697.36
2018-08-31  7379.15  -2396.18  4982.97
2018-09-30  5234.71  -3286.99  1947.72
2018-10-31  5022.23  -2848.35  2173.88
2018-11-30  6018.96  -2963.65  3055.31
2018-12-31  5635.10  -3427.99  2207.11
2019-01-31  4769.44  -5187.31  -417.87
2019-02-28  4500.01  -3163.40  1336.61
2019-03-31  7792.25  -3241.51  4550.74
2019-04-30  5931.89  -4829.55  1102.34
2019-05-31  5341.01  -4673.51   667.50
2019-06-30  4514.35 -11999.60 -7485.25
2019-07-31  5642.40  -4148.06  1494.34
2019-08-31  7304.10  -4266.17  3037.93
2019-09-30  6537.06  -5998.42   538.64
